# Pipeline Silver → Gold

Modelagem dimensional da camada Gold: dimensões, fato e tabelas ponte a partir dos dados limpos da Silver.

In [0]:
# Criação do catálogo e schema da camada Gold
spark.sql("CREATE CATALOG IF NOT EXISTS cinedata_lakehouse")
spark.sql("CREATE SCHEMA IF NOT EXISTS cinedata_lakehouse.gold")
spark.sql("USE CATALOG cinedata_lakehouse")
spark.sql("USE SCHEMA gold")

# Dimensão: dim_movies

Criação da dimensão de filmes com surrogate key baseada em hash (SHA-256) para idempotência.

In [0]:
from pyspark.sql.functions import col, sha2

# Leitura da Silver 
df_info = spark.read.table("cinedata_lakehouse.silver.tb_info_filmes")

# Criação da dim_movies com Surrogate Key baseada em Hash
df_dim_movies = df_info.select(
    sha2(col("id_filme").cast("string"), 256).alias("sk_movie_id"),
    col("id_filme"),
    col("titulo"),
    col("data_lancamento"),
    col("ano_lancamento"),
    col("duracao_minutos"),
    col("idioma_original"),
    col("status_filme"),
    col("sinopse")
)

# Persistência com FQN
(
    df_dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cinedata_lakehouse.gold.dim_movies")
)

# Dimensão: dim_genres

Extração de gêneros únicos da Silver com surrogate key por hash.

In [0]:
from pyspark.sql.functions import col, sha2

df_generos_silver = spark.read.table("cinedata_lakehouse.silver.tb_generos")

df_dim_genres = df_generos_silver.select("nome_genero").distinct()

df_dim_genres = df_dim_genres.withColumn(
    "sk_genre_id", 
    sha2(col("nome_genero"), 256)
).select("sk_genre_id", "nome_genero")

(
    df_dim_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cinedata_lakehouse.gold.dim_genres")
)

# Dimensões: dim_people e dim_companies

Dimensões de pessoas (atores, diretores, roteiristas) e produtoras, com SK composta por nome + papel para garantir unicidade.

In [0]:
from pyspark.sql.functions import col, sha2, concat_ws

df_pessoas_empresas = spark.read.table("cinedata_lakehouse.silver.tb_pessoas_empresas")

# 1. dim_people (Pessoas Físicas)
df_dim_people = df_pessoas_empresas.filter(
    col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
).select(
    col("nome_entidade").alias("nome_pessoa"),
    col("tipo_entidade").alias("tipo_pessoa")
).distinct()


df_dim_people = df_dim_people.withColumn(
    "sk_person_id", 
    sha2(concat_ws("|", col("nome_pessoa"), col("tipo_pessoa")), 256) 
).select("sk_person_id", "nome_pessoa", "tipo_pessoa")

(
    df_dim_people.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("cinedata_lakehouse.gold.dim_people")
)

# 2. dim_companies (Produtoras)
df_dim_companies = df_pessoas_empresas.filter(
    col("tipo_entidade") == "Produtora"
).select(
    col("nome_entidade").alias("nome_produtora")
).distinct()

df_dim_companies = df_dim_companies.withColumn(
    "sk_company_id", 
    sha2(col("nome_produtora"), 256)
).select("sk_company_id", "nome_produtora")

(
    df_dim_companies.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("cinedata_lakehouse.gold.dim_companies")
)

# Tabela: dim_reviews

Agregação de avaliações dos usuários por filme (quantidade e nota média). Implementado como 'dimensão' para cumprir a documentação oficial do projeto.

In [0]:
from pyspark.sql.functions import col, count, round, avg, sha2

df_avaliacoes = spark.read.table("cinedata_lakehouse.silver.tb_avaliacoes_usuarios")

# Agregação por id_filme 
df_reviews_agg = df_avaliacoes.groupBy("id_filme").agg(
    count("*").alias("qtd_avaliacoes_usuarios"),
    round(avg("nota_usuario"), 2).alias("nota_media_usuarios")
)

df_dim_movies_sk = spark.read.table("cinedata_lakehouse.gold.dim_movies").select("id_filme", "sk_movie_id")

df_dim_reviews = df_reviews_agg.join(
    df_dim_movies_sk, on="id_filme", how="inner"
).withColumn(
    "sk_review_id", sha2(col("id_filme").cast("string"), 256)
).select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")

(
    df_dim_reviews.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("cinedata_lakehouse.gold.dim_reviews")
)

# Fato: fact_movies_performance

Tabela fato com métricas financeiras e de engajamento por filme, via left join com a Silver.

In [0]:
from pyspark.sql.functions import col

df_fin = spark.read.table("cinedata_lakehouse.silver.tb_financeiro_filmes")
df_met = spark.read.table("cinedata_lakehouse.silver.tb_metricas_engajamento")
df_movies = spark.read.table("cinedata_lakehouse.gold.dim_movies").select("id_filme", "sk_movie_id")

# A Silver já deduplicou estes dados, prevenindo fan-out neste left join.
df_fact = (
    df_movies
    .join(df_fin, on="id_filme", how="left")
    .join(df_met, on="id_filme", how="left")
)

df_fact_performance = df_fact.select(
    col("sk_movie_id"),
    col("orcamento_usd"),
    col("receita_usd"),
    col("lucro_usd"),
    col("orcamento_brl"),
    col("receita_brl"),
    col("lucro_brl"),
    col("popularidade"),
    col("nota_media_tmdb"),
    col("qtd_votos_tmdb"),
    col("nota_media_imdb"),
    col("qtd_votos_imdb")
)

(
    df_fact_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cinedata_lakehouse.gold.fact_movies_performance")
)

# Tabelas Ponte (Bridge)

Tabelas de associação many-to-many entre dim_movies e dim_genres, dim_people e dim_companies.

In [0]:
from pyspark.sql.functions import col

df_movies_sk = spark.read.table("cinedata_lakehouse.gold.dim_movies").select("id_filme", "sk_movie_id")

# bridge_movie_genre
df_generos_silver = spark.read.table("cinedata_lakehouse.silver.tb_generos")
df_dim_genres = spark.read.table("cinedata_lakehouse.gold.dim_genres")

df_bridge_genre = (
    df_generos_silver
    .join(df_movies_sk, on="id_filme", how="inner")
    .join(df_dim_genres, on="nome_genero", how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

(
    df_bridge_genre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cinedata_lakehouse.gold.bridge_movie_genre")
)

# bridge_movie_person
df_pessoas_empresas = spark.read.table("cinedata_lakehouse.silver.tb_pessoas_empresas")
df_dim_people = spark.read.table("cinedata_lakehouse.gold.dim_people")

df_silver_people = df_pessoas_empresas.filter(col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))

# Join direto com a dim_people pelas chaves naturais 
df_bridge_person = (
    df_silver_people
    .join(df_movies_sk, on="id_filme", how="inner")
    .join(
        df_dim_people, 
        (df_silver_people.nome_entidade == df_dim_people.nome_pessoa) & 
        (df_silver_people.tipo_entidade == df_dim_people.tipo_pessoa), 
        how="inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

(
    df_bridge_person.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cinedata_lakehouse.gold.bridge_movie_person")
)

# bridge_movie_company
df_silver_companies = df_pessoas_empresas.filter(col("tipo_entidade") == "Produtora")
df_dim_companies = spark.read.table("cinedata_lakehouse.gold.dim_companies")

df_bridge_company = (
    df_silver_companies
    .join(df_movies_sk, on="id_filme", how="inner")
    .join(
        df_dim_companies, 
        df_silver_companies.nome_entidade == df_dim_companies.nome_produtora, 
        how="inner"
    )
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

(
    df_bridge_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cinedata_lakehouse.gold.bridge_movie_company")
)

# Contexto GenAI

Geração de documentos contextuais para LLM a partir das tabelas dimensionais e fato da camada Gold.

In [0]:
from pyspark.sql.functions import col, concat, lit, coalesce, collect_list, concat_ws, sort_array

# Leituras essenciais
df_movies = spark.read.table("cinedata_lakehouse.gold.dim_movies")
df_fact = spark.read.table("cinedata_lakehouse.gold.fact_movies_performance")
df_bridge_person = spark.read.table("cinedata_lakehouse.gold.bridge_movie_person")
df_dim_people = spark.read.table("cinedata_lakehouse.gold.dim_people")

# 1. Base principal
df_base = df_movies.join(df_fact, on="sk_movie_id", how="left")

# 2. Agregação do Elenco e Diretores (Determinística com sort_array)
df_movie_people = df_bridge_person.join(df_dim_people, on="sk_person_id", how="inner")

df_atores = (
    df_movie_people.filter(col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(concat_ws(", ", sort_array(collect_list("nome_pessoa"))).alias("elenco_agregado"))
)

df_diretores = (
    df_movie_people.filter(col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(concat_ws(", ", sort_array(collect_list("nome_pessoa"))).alias("diretores_agregados"))
)

# 3. Consolidação 
df_final = (
    df_base
    .join(df_atores, on="sk_movie_id", how="left")
    .join(df_diretores, on="sk_movie_id", how="left")
)

# 4. Concatenação Protegida com Fallbacks
df_genai_context = df_final.withColumn(
    "llm_context_document",
    concat(
        lit("O filme "), coalesce(col("titulo"), lit("Sem Título")),
        lit(", lançado no ano de "), coalesce(col("ano_lancamento").cast("string"), lit("um ano não informado")),
        lit(", faturou "), coalesce(concat(lit("US$ "), col("receita_usd").cast("string")), lit("um valor não divulgado")),
        lit(" e teve um custo de "), coalesce(concat(lit("US$ "), col("orcamento_usd").cast("string")), lit("um orçamento não divulgado")),
        lit(". Estrelado por "), coalesce(col("elenco_agregado"), lit("um elenco não detalhado na base")),
        lit(" e dirigido por "), coalesce(col("diretores_agregados"), lit("um diretor desconhecido")),
        lit(", o filme possui a seguinte sinopse: "), coalesce(col("sinopse"), lit("Não disponível."))
    )
).select(
    col("id_filme").alias("movie_id"),
    col("titulo").alias("title"),
    col("llm_context_document")
)

# Persistência 
(
    df_genai_context.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cinedata_lakehouse.gold.gold_genai_movies_context")
)

In [0]:
from pyspark.sql.functions import col

print("=" * 60)
print("VALIDAÇÃO DE QUALIDADE DE DADOS — Camada Gold")
print("=" * 60)

# 1. Contagem de linhas por tabela
tabelas_gold = [
    "dim_movies", "dim_genres", "dim_people", "dim_companies",
    "dim_reviews", "fact_movies_performance",
    "bridge_movie_genre", "bridge_movie_person", "bridge_movie_company",
    "gold_genai_movies_context"
]

print("\n— Contagem de linhas —")
for t in tabelas_gold:
    n = spark.read.table(f"cinedata_lakehouse.gold.{t}").count()
    print(f"  {t}: {n} linhas")

# 2. Verificação de SKs nulas (dimensões)
dims_sk = {
    "dim_movies": "sk_movie_id",
    "dim_genres": "sk_genre_id",
    "dim_people": "sk_person_id",
    "dim_companies": "sk_company_id",
}

print("\n— SKs nulas (dimensões) —")
for t, sk in dims_sk.items():
    n = spark.read.table(f"cinedata_lakehouse.gold.{t}").filter(col(sk).isNull()).count()
    print(f"  {'✓' if n == 0 else '✗'} {t}.{sk}: {n} nulos")

# 3. Verificação de SKs duplicadas (dimensões)
print("\n— SKs duplicadas (dimensões) —")
for t, sk in dims_sk.items():
    dup = (spark.read.table(f"cinedata_lakehouse.gold.{t}")
           .groupBy(sk).count().filter(col("count") > 1).count())
    print(f"  {'✓' if dup == 0 else '✗'} {t}.{sk}: {dup} duplicadas")

# 4. Verificação de chaves compostas (tabelas ponte)
print("\n— Chaves compostas duplicadas (tabelas ponte) —")
bridges = {
    "bridge_movie_genre": ["sk_movie_id", "sk_genre_id"],
    "bridge_movie_person": ["sk_movie_id", "sk_person_id"],
    "bridge_movie_company": ["sk_movie_id", "sk_company_id"],
}
for t, keys in bridges.items():
    dup = (spark.read.table(f"cinedata_lakehouse.gold.{t}")
           .groupBy(*keys).count().filter(col("count") > 1).count())
    print(f"  {'✓' if dup == 0 else '✗'} {t}: {dup} duplicadas")

# 5. Verificação de contexto GenAI
print("\n— Contexto GenAI —")
df_ctx = spark.read.table("cinedata_lakehouse.gold.gold_genai_movies_context")
n_nulls_ctx = df_ctx.filter(col("llm_context_document").isNull()).count()
n_title_nulls = df_ctx.filter(col("title").isNull()).count()
print(f"  {'✓' if n_nulls_ctx == 0 else '✗'} llm_context_document nulo: {n_nulls_ctx}")
print(f"  {'✓' if n_title_nulls == 0 else '✗'} title nulo: {n_title_nulls}")

print("\n" + "=" * 60)
print("Validação concluída.")